# RiskSense AI — Forecast Model Training

This notebook trains a Random Forest model to predict risk scores
at 6-hour and 12-hour horizons using engineered features.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

In [ ]:
from src.model.train import run as train_model
result = train_model()
print(result)

In [ ]:
from src.model.predict import run as predict_risk
pred_df = predict_risk()
pred_df.head()

In [ ]:
if 'predicted_risk_6h' in pred_df.columns and 'risk_score' in pred_df.columns:
    valid = pred_df[['risk_score', 'predicted_risk_6h']].dropna()
    if len(valid) > 0:
        mae = mean_absolute_error(valid['risk_score'], valid['predicted_risk_6h'])
        rmse = np.sqrt(mean_squared_error(valid['risk_score'], valid['predicted_risk_6h']))
        r2 = r2_score(valid['risk_score'], valid['predicted_risk_6h'])
        print(f'6h Forecast — MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}')

In [ ]:
import plotly.graph_objects as go

if 'risk_score' in pred_df.columns and 'predicted_risk_6h' in pred_df.columns:
    dfp = pred_df.sort_values('timestamp').reset_index(drop=True)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=dfp['risk_score'].values[:50],
        mode='lines+markers', name='Actual Risk',
        line=dict(color='orange', width=2)
    ))
    fig.add_trace(go.Scatter(
        y=dfp['predicted_risk_6h'].values[:50],
        mode='lines+markers', name='Predicted 6h',
        line=dict(color='red', width=2, dash='dash')
    ))
    fig.update_layout(
        title='Actual vs Predicted Risk (first 50 time steps)',
        xaxis_title='Time Step',
        yaxis_title='Risk Score',
        hovermode='x unified'
    )
    fig.show()

## Notes
- The model training requires at least 10 rows of historical risk data.
- Features include hazard counts, weather variables, and time features.
- Training more data over longer periods will improve accuracy.